# V1
来自baseline, 针对测试得到的问题进行处理分析

**问题**：md每个都是菜谱，南派红烧肉，徽派红烧肉，家常红烧肉，简易红烧肉， 每个文档分成很多快。 当我输入红烧肉选择时候，因为知识库有limit，所以只能感知到一些包含红烧肉居多的chunk。 用户问选择哪种红烧肉做法，实际只有4选一，感知不到宏观东西。

**方法一**：
- 增加全文文档向量，与分块 建立父子关系。 检索到chunk后与父亲文档一起带回 构造上下文。
    - 疑问： 这可能会产生很长内容，且父亲内容由于与子chunk重复，context会有一些重复信息，这应该是没有用的。而且这可能回归到刚开始直接不分块。 
- 因为知识库是topk检索，还是会有一部分没有找到。 
    - 疑问： 虽然可能还不够，但这应该是允许的。

方法二：
知识图谱。（未实现）

此外，我们引入稀疏向量，允许对菜谱种一些定量信息保持准确 敏感 完整

基于baseline代码修改

## baseline不变代码

In [80]:
import torch
import glob
import os
from dotenv import load_dotenv
import os
from typing import Any
import uuid

cuda_available = torch.cuda.is_available()
print(f"是否支持 CUDA (GPU加速): {cuda_available}")

load_dotenv()
print(os.getcwd())

是否支持 CUDA (GPU加速): False
/home/dong/data-analysis/myrag/cookrag


In [81]:
COLLECTION_NAME = 'cookrag_ct_v1' # 新的知识库

In [82]:
from pymilvus.model.hybrid import BGEM3EmbeddingFunction
embed_encoder = BGEM3EmbeddingFunction( 
    model_name='BAAI/bge-small-zh-v1.5', # Specify the model name
    device='cpu', # Specify the device to use
    use_fp16=False # 使用16位精度，加快速度
    )

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 39397.36it/s]


In [83]:
from pymilvus import MilvusClient
client = MilvusClient(uri='milvus_cookrag.db')

In [84]:
def build_context(knowledges: list[dict[str, Any]])->str:
    """ 知识库返回结果转换位上下文 """
    context = ""
    for kd in knowledges:
        context+= kd['entity']['content']
        print(f'distance:{kd['distance']:.4f}, content:{kd['entity']['content']}')
    return context

In [85]:
from langchain_core.prompts import ChatPromptTemplate
template = ChatPromptTemplate(
    messages=[
        ('system', '你是一个专业的烹饪饮食专家, 严格根据上下文信息回答，如果不知道就说不知道'),
        ('human', """
        上下文信息：{context} 
                
        用户输入:{query}
        """)
            ]
)

In [86]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    api_key= os.getenv('AIHUBMIX_API_KEY'),
    base_url="https://aihubmix.com/v1",
    model='xiaomi-mimo-v2-omni-free',
) 

In [87]:
chain = template | llm

In [88]:
def rag(query:str)->str:
    """ 用户接口 """
    knowledges = knowledge_search(user_query=query)
    context = build_context(knowledges=knowledges)
    response = chain.invoke({'context': context, 'query':query})
    return response.content

In [89]:
from IPython.display import display, Markdown
def display_md_jupyter(s:str):
    """ jupyter是纯文本输出，使用IPython交互内置渲染md """
    display(Markdown(s))


## 修改

我们在`metadata`字段中，为每个全文文档设置`id`,为每个chunk子文档设置`parent_id`,这两个相等关联

In [90]:
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader

def load_documents() -> list[Document]:
    """ 将每个md文档加载为对应document结构 ，添加id辅助后面关联"""
    documents = []
    for md_file in glob.glob(os.path.join(os.getcwd(), 'data','*.md')):
        loader = TextLoader(file_path=md_file)
        data = loader.load()
        data = data[0]
        data.metadata['id'] = str(uuid.uuid4())
        documents.append(data)
    return documents

In [91]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain.text_splitter import RecursiveCharacterTextSplitter

def splitter_documents(documents: list[Document]) -> list[Document]:
    """ 对一些document分块 """
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]
    markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
    
    text_splitter = RecursiveCharacterTextSplitter(
        separators = ["\n\n", "\n", "。", "，", " ", ""],  # 分隔符优先级
        chunk_size = 200,
        chunk_overlap=10
    )
    all_sections = []
    
    for doc in documents:
        sections  = markdown_splitter.split_text(doc.page_content)
        for section in sections:
            section.metadata['parent_id'] = doc.metadata['id']
        all_sections.extend(sections)

    chunks = text_splitter.split_documents(all_sections)

    # page_content会丢失层级信息，只保留内容. 
    # 这导致一勺盐不知道是红烧肉还是豆腐
    # 需要手动为其注入
    for chunk in chunks:
        prefix = ""
        h1 = chunk.metadata.get('Header 1', '无')
        h2 = chunk.metadata.get('Header 2', '无')
        h3 = chunk.metadata.get('Header 3', '无')
        prefix = f"主题: {h1} > 章节: {h2} > 细节: {h3}\n内容: "
        chunk.page_content = prefix + chunk.page_content 
    return chunks

In [ ]:
from pymilvus import MilvusClient,FieldSchema, CollectionSchema,DataType
def setup_collection():
    if client.has_collection(COLLECTION_NAME):
        client.drop_collection(COLLECTION_NAME) 
    # 定义数据库格式
    fileds = [
        FieldSchema(name='id', dtype=DataType.INT64, is_primary=True,auto_id = True),
        FieldSchema(name='content', dtype=DataType.VARCHAR, max_length=1024),
        FieldSchema(name='metadata', dtype=DataType.JSON),
        FieldSchema(name='sparse_vector', dtype=DataType.SPARSE_FLOAT_VECTOR),
        FieldSchema(name='dense_vector', dtype=DataType.FLOAT_VECTOR, dim=embed_encoder.dim['dense']),
    ]
    schema = CollectionSchema(fields=fileds, description='cookrag')
    client.create_collection(collection_name=COLLECTION_NAME, schema=schema)
    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name='dense_vector',
        index_type='IVF_FLAT', # 索引类型
        metric_type='IP',  # 
    )
    client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)

    index_params = client.prepare_index_params()
    index_params.add_index(
        field_name='sparse_vector',
        index_type='SPARSE_INVERTED_INDEX', # 索引类型
        metric_type='IP',  # 必须显式指定为内积

    )
    client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)

    client.describe_collection(collection_name=COLLECTION_NAME)

In [93]:
from scipy.sparse import coo_array, coo_matrix
def format_sparse_vector_to_dict(coo_arr: coo_array) -> dict:
    """ 
    将 coo_array 数组 转换为 Milvus 稀疏向量字段要求的字典格式: 
    {index: value}
    """
    # 1D coo_array 的坐标在 coords[0] 中
    indices = coo_arr.coords[0] 
    return dict(zip(indices, coo_arr.data))


In [94]:
def insert_collection(documents:list[Document])->None:
    """ 把docuemnts 放到向量数据库 """
    data_to_insert = []

    for doc in documents:
        embeddings = embed_encoder([doc.page_content])
        data_to_insert.append({
            'metadata': doc.metadata,
            'dense_vector': embeddings['dense'][0],
            'sparse_vector': format_sparse_vector_to_dict(embeddings['sparse'][0]),
            'content': doc.page_content
        })
    client.insert(
        collection_name=COLLECTION_NAME,
        data = data_to_insert
    )

In [95]:
def build_collection():
    """ 构建向量知识库 """
    setup_collection()
    print('collection 结构已准备好')
    documents = load_documents()
    splitted_documents = splitter_documents(documents)
    insert_collection(documents)
    print('插入全文文档向量 ok')
    insert_collection(splitted_documents)
    print('插入文本块ok')

In [ ]:
build_collection() 

collection 结构已准备好


You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
I0000 00:00:1777731427.696369    8657 chttp2_transport.cc:1182] unix:/tmp/tmpai5t93rp_milvus_cookrag.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {created_time:"2026-05-02T22:17:07.696356411+08:00", http2_error:11, grpc_status:14}
E0000 00:00:1777731427.696575    8657 chttp2_transport.cc:1210] unix:/tmp/tmpai5t93rp_milvus_cookrag.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


插入全文文档向量 ok
插入文本块ok


In [ ]:
from pymilvus import RRFRanker,AnnSearchRequest
def knowledge_search(user_query:str) -> list[dict[str, Any]]:
    """ 混合检索 稀疏向量和稠密向量 """
    query_embeddings = embed_encoder([user_query])
    dense_vec, sparse_vec = query_embeddings['dense'][0], format_sparse_vector_to_dict(query_embeddings['sparse'][0])
    # RRF
    rerank = RRFRanker(k=60) # 会融合两个结果，得到, k=60表示平滑参数（就是两个结果平衡参数）
    dense_req = AnnSearchRequest(
        [dense_vec],
        anns_field='dense_vector',
        limit=30,
        param={"metric_type": "IP"}
    )
    sparse_req = AnnSearchRequest(
        [sparse_vec],
        anns_field='sparse_vector',
        limit=30,
        param={"metric_type": "IP"}
    )
    results = client.hybrid_search(
        collection_name=COLLECTION_NAME,
        reqs=[dense_req, sparse_req],
        ranker=rerank,
        limit=30,
        output_fields=['content','metadata']
    )[0]
    seen_ids = {r['id'] for r in results}
    need_ids = set()
    for r in results:
        parent_id = r['metadata'].get('parent_id', '')
        if parent_id and parent_id not in seen_ids:
            need_ids.add(parent_id)
    if need_ids:
        pid_list = list(need_ids)

        # 准确查询
        parent_results = client.query(
            collection_name=COLLECTION_NAME,
            filter=f"metadata['id'] in {pid_list}",
            output_fields=['content','metadata']

        )
        # 和混合检索得出的格式不一样，统一下。
        formatted_parents = []
        for p in parent_results:
            formatted_parents.append({
                'id': p['id'],
                'distance': 0.0, # 补齐字段，防止后续代码报错
                'entity': {
                    'content': p['content'],
                    'metadata': p['metadata']
                }
            })
        results.extend(formatted_parents)
    return results

In [132]:
results = knowledge_search(user_query='红烧肉的做法')

In [ ]:
def sparse_vec_info(s:str):
    """ from ai. 描述稀疏向量激活了多少关键词 """
    embeddings = embed_encoder([s])
    sparse_obj = embeddings['sparse'][0]
    print(f"类型: {type(sparse_obj)}")
    print(f"非零元素个数: {sparse_obj.nnz}") # 查看有多少个关键词被激活
    tokenizer = embed_encoder.model.tokenizer
    coo = sparse_obj.tocoo()
    token_weights = sorted(zip(coo.col, coo.data), key=lambda x: x[1], reverse=True)
    
    print(f"{'关键词':<10} | {'权重':<10}")
    print("-" * 25)
    for token_id, weight in token_weights:
        token_text = tokenizer.decode([token_id]) # 将 ID 转回文字
        print(f"{token_text:<10} | {weight:.4f}")

可以看到，对于短查询，激活的关键词很少很少。大部分只是把每个字拆开。  这完全不包含信息，是完全的噪声

- 我们发现，对于红烧肉稀疏向量，只有3个非0。 对于一大段话，大部分都是单字，且产生很多非0.
- 做度量时候，内积，就是在这 红 烧 肉 三个字进行。 
- 所以文字越多的文章，越可能相近的。

所以我们会设置更多limit，让RRF过滤。 
RRF是统计模型，所以不能无限制增大limit,会增大噪声。

西红柿对于 红烧肉 的稀疏得分很高， 这是可以理解的。是允许的。

如果需要更改双方得分比重：rerank = WeightedRanker(0.8, 0.2) 

## 测试

In [ ]:
display_md_jupyter(rag('对比一下南派和徽派红烧肉在甜度控制上的区别。'))

distance:0.0164, content:主题: 南派红烧肉的做法 > 章节: 操作 > 细节: 无
内容: - 打开锅盖，待汤汁快没有的时粘稠状出锅（切记不可收干）；
distance:0.0164, content:主题: 利提巧卡的做法 > 章节: 附加内容 > 细节: 无
内容: - 烤鹰嘴豆粉(Sattu)是比哈尔邦的特色食材，营养价值极高，富含蛋白质和纤维。在中国可能较难购买，可以自制：将鹰嘴豆(Chana)干炒至深棕色后磨成粉。
- 传统的利提用牛粪饼火(Cow dung cake fire)烤制，赋予独特的烟熏风味。家用烤箱或明火可以替代。
- 芥末油是这道菜的灵魂，不建议替换。如果买不到芥末油，可以使用菜籽油作为次选。
distance:0.0161, content:主题: 懒人蛋挞的做法 > 章节: 无 > 细节: 无
内容: ![蛋挞成品](./懒人蛋挞.png)  
蛋挞是一道常见的可口甜品，通常而言制作蛋挞是需要调和蛋挞液和制作蛋挞皮的，这个过程比较复杂和耗时，但是网购半成品恰恰解决解决以上的难题，初学者只需大约 40 分就可以完成。从今往后只要家里有烤箱，就可以化身烘焙达人，帮家人烤蛋挞！  
预估烹饪难度：★★★
distance:0.0161, content:主题: 徽派红烧肉的做法 > 章节: 无 > 细节: 无
内容: 徽式红烧肉是一道由五花肉等食材制成的菜肴。  
预估烹饪难度：★★★★
distance:0.0159, content:主题: 使用空气炸锅 > 章节: 什么是空气炸锅 > 细节: 工作方式
内容: 空气炸锅借由上方的加热器产生高温热风，让热空气在食物周遭循环对流，快速加热食物自身的油脂，带走食物的水分，产生油炸的效果，并创造类似油炸食物的酥脆感。
distance:0.0159, content:主题: 南派红烧肉的做法 > 章节: 操作 > 细节: 无
内容: - 建议先拿出来一半葱姜，再将剩下的`生姜、葱白、蒜、花椒、八角、香叶`提前放入一个碗中备用
- 凉水锅中放入切好的五花肉，加入料酒与 2/5 葱姜，煮 15 分钟去掉血腥，捞出来后洗干净；
- 炒[糖色](./../../condiment/简易版炒糖色.md)，注意采用其中提到的操作 2 来制作糖色。
distance:0.0156, 

根据提供的菜谱上下文，南派红烧肉和徽派红烧肉在甜度控制上有显著区别，主要体现在**糖的种类、用量和添加目的**上。

### 南派红烧肉的甜度控制
*   **糖的种类与用量**：食谱中明确使用了**冰糖（约15块）** 和**白砂糖（30g）**。冰糖用于炒糖色，提供红亮的色泽和基础甜味；额外的白砂糖则用于直接补充甜度。
*   **操作与目的**：甜度主要通过两个步骤控制：
    1.  **炒糖色**：使用冰糖，通过美拉德反应和焦糖化作用，主要目的是**上色**和产生复合风味，甜味是附带产生且较为醇厚。
    2.  **直接调味**：在炒糖色后，可能通过加入的白砂糖（30g）来**直接调节和补足**菜肴最终的甜度。
*   **特点**：甜味来源**双重**（糖色+直接添加），甜度相对**温和、有层次**，旨在平衡咸鲜，形成咸甜交融的复合味。

### 徽派红烧肉的甜度控制
*   **糖的种类与用量**：食谱中仅使用**白砂糖**，且用量高达**100g**（远超南派的冰糖+白砂糖总量）。
*   **操作与目的**：甜度主要通过一个核心步骤控制：
    1.  **大量白砂糖炒糖色**：将50ml油与100g白砂糖一同翻炒至咖啡色。这里的糖色步骤**同时承担了上色和提供主要甜度**的双重任务。由于糖的用量极大，炒制后形成的焦糖风味会非常浓郁，甜味也十分突出。
*   **特点**：甜味来源**单一但极其浓郁**。大量的白砂糖在炒制后赋予了菜肴非常**直接、鲜明且主导性**的甜味，甜度感明显高于南派做法。

### 总结对比
| 特征 | 南派红烧肉 | 徽派红烧肉 |
| :--- | :--- | :--- |
| **主要用糖** | 冰糖（炒色） + 白砂糖（调味） | 仅白砂糖（大量） |
| **甜度来源** | 1. 炒糖色（冰糖）<br>2. 直接添加（白砂糖） | 1. 大量白砂糖炒制的糖色（承担上色与主要甜味） |
| **甜度表现** | **温和、平衡**，作为复合味的一部分。 | **鲜明、浓郁、主导**，是风味的核心特征之一。 |
| **控制逻辑** | 通过**两种糖分阶段添加**来精细调节色泽与口味的平衡。 | 通过**单一糖类的大剂量使用**来确立强烈的风味基调。 |

**结论**：南派红烧肉的甜度控制更侧重于**平衡与层次**，甜味相对含蓄；而徽派红烧肉的甜度控制则追求**直接与浓郁**，甜味是其最鲜明的标志之一。

😊😊😊
- 可以看到，相比于之前，有了丰富的上下文。因此可以回答
- 用量明确，稀疏向量也起了作用

In [137]:
display_md_jupyter(rag('我想做红烧肉，但我家只有白砂糖没有冰糖，哪种做法最适合我'))

I0000 00:00:1777733200.809867    8657 chttp2_transport.cc:1182] unix:/tmp/tmpai5t93rp_milvus_cookrag.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {grpc_status:14, http2_error:11, created_time:"2026-05-02T22:46:40.809852691+08:00"}
E0000 00:00:1777733200.810016    8657 chttp2_transport.cc:1210] unix:/tmp/tmpai5t93rp_milvus_cookrag.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


distance:0.0164, content:主题: 简易红烧肉的做法 > 章节: 操作 > 细节: 开始制作
内容: - 冷水锅中放入切好的`猪五花肉`，加入料酒与葱姜，煮 15 分钟去掉血腥
- 锅中放入两片`生姜`提味
- 开中小火后直接加入`五花肉`，不需要放入食用油，每块`五花肉`六个面都煎一下，煎至出油即可
- 将煎出的油倒出备用，并将`五花肉`推至一边，加入 15g `冰糖`，翻炒至`冰糖`融化；
- 融化后将五花肉与冰糖炒至融合上色，加入
- `生抽` 10ml
- `老抽` 15ml
distance:0.0161, content:主题: 简易红烧肉的做法 > 章节: 计算 > 细节: 无
内容: 每次制作前需要确定计划做几份。一份正好够 2-3 个人吃。  
如果只有 1 人食用，可以考虑食材减半。  
- 猪五花肉：约 3~4 斤
- 姜： 6 片
- 冰糖：15 克（约 7 块）
- 生抽：10ml
- 老抽：15ml
- 料酒：5ml
- 开水：没过食材的量，需要 600ml-900ml
- 香叶：3 片
- 八角：2 个
distance:0.0159, content:主题: 简易红烧肉的做法 > 章节: 操作 > 细节: 开始制作
内容: - 加入 2-3g `盐`，翻炒一下，就可以出锅了。
distance:0.0156, content:主题: 湖南家常红烧肉的做法 > 章节: 操作 > 细节: 无
内容: * 带皮五花肉洗净冷水下锅，加入姜片 2~3 片去腥味，煮到沸腾捞出冷水冲净白沫
* 五花肉切块，尺寸 1.5cm*1.5cm 块状大小
* 热锅加入油，加入冰糖小火搅拌至焦糖色即可，加入切好的五花肉，中火翻炒上色
* 加入备好的姜片、八角、桂皮、生抽、老抽、料酒、干小米椒、盐，小火翻炒 1 分钟，加开水没过肉
* 加盖中火煮沸，转小火慢顿 30 分钟，慢炖期间，间隔 10 分钟搅拌一次防止粘锅
distance:0.0154, content:主题: 南派红烧肉的做法 > 章节: 操作 > 细节: 无
内容: - 建议先拿出来一半葱姜，再将剩下的`生姜、葱白、蒜、花椒、八角、香叶`提前放入一个碗中备用
- 凉水锅中放入切好的五花肉，加入料酒与 2/5 葱姜，煮 15 分钟去掉血腥，捞出来后洗干净；
- 炒[糖色](./

根据提供的菜谱信息，**徽派红烧肉的做法**最适合您，因为它明确使用白砂糖作为核心原料。

**原因如下：**
1.  **原料匹配**：在“徽派红烧肉的做法”的“必备原料和工具”和“计算”部分，明确列出的糖是 **“白砂糖”**（100克），而非冰糖。
2.  **操作明确**：其操作步骤为“锅中加入 50 ml 食用油，**倒入白砂糖，翻炒到咖啡色**”，这是一个标准的油炒糖色流程，完全使用白砂糖。

其他菜谱（如简易红烧肉、湖南家常红烧肉）在原料和操作中均指定使用“冰糖”。虽然“简易版炒糖色”教程中提到可以用白砂糖，但那是通用技巧，而非特定红烧肉菜谱的既定步骤。

因此，直接遵循 **“徽派红烧肉的做法”** 即可，无需替换或调整糖的种类。

😊😊😊
回答是合理的

In [138]:
display_md_jupyter(rag('红烧肉怎么上色'))

distance:0.0164, content:主题: 南派红烧肉的做法 > 章节: 操作 > 细节: 无
内容: - 建议先拿出来一半葱姜，再将剩下的`生姜、葱白、蒜、花椒、八角、香叶`提前放入一个碗中备用
- 凉水锅中放入切好的五花肉，加入料酒与 2/5 葱姜，煮 15 分钟去掉血腥，捞出来后洗干净；
- 炒[糖色](./../../condiment/简易版炒糖色.md)，注意采用其中提到的操作 2 来制作糖色。
distance:0.0161, content:主题: 简易红烧肉的做法 > 章节: 操作 > 细节: 开始制作
内容: - 加入 2-3g `盐`，翻炒一下，就可以出锅了。
distance:0.0159, content:主题: 简易红烧肉的做法 > 章节: 操作 > 细节: 开始制作
内容: - `料酒` 5ml
- 翻炒至上色；
- 加入`烧好的开水`炖煮 40 分钟（刀工差的同学切的过大请自觉延长炖煮时间），并放入
- `生姜` 2 片
- `香叶` 3 片
- `八角` 2 个
- 盖上锅盖煮至沸腾后，加入煮好扎好孔的`鹌鹑蛋`和`豆皮`，开中小火，等待 40 分钟。（中途可适当翻搅防止粘锅）；
- 打开锅盖，待汤汁快没有的时候开大火收汁（切记不可收干）；
distance:0.0156, content:主题: 简易红烧肉的做法 > 章节: 必备原料和工具 > 细节: 无
内容: - 注：如果有可能，请尽量把刀磨的锋利一些。
- 主料：`大肉`、`鸡蛋`（可选）、`豆皮`（可选）
- 辅料：`生姜`、`冰糖`、`生抽`、`老抽`、`料酒`、`香叶`、`八角`、`盐`、`水`、`葱`（记得要开水）
distance:0.0154, content:主题: 南派红烧肉的做法 > 章节: 操作 > 细节: 无
内容: 建议在此之前查看炒[糖色](./../../condiment/简易版炒糖色.md)教程  
- `猪五花肉`切大块（约 4.5cm ）
- `生姜`切片（每片厚度约 3mm ）
- `开水`烧开
- `凉水`自来水即可
- `小葱`小葱白色的部分`葱白`切成小段（小葱最佳，大葱也可以）
- `蒜`中间切开，不要拍扁，否则难以捞出以至最后`收汁`时影响味道
distance:0.0152, content:主题: 简易红烧肉

根据提供的四种红烧肉做法，红烧肉上色主要通过以下两种方法，通常结合使用：

### 1. 炒糖色
这是上色的核心步骤，不同做法操作略有差异：
*   **南派做法**：需专门“炒糖色”，建议参考“简易版炒糖色”教程中的操作2。之后将五花肉与糖色翻炒融合上色。
*   **简易做法**：锅中不放油，直接煎五花肉至出油，推至一边后加入**冰糖**翻炒至融化，再与五花肉混合翻炒上色。
*   **湖南家常做法**：热锅加油，放入**冰糖**小火搅拌至焦糖色，然后加入五花肉翻炒上色。
*   **徽派做法**：锅中加油，倒入**白砂糖**翻炒至咖啡色，再倒入煎过的五花肉翻炒上色。

### 2. 加酱油（生抽和老抽）
在炒糖色后或炖煮前，加入酱油辅助上色和调味：
*   **南派做法**：在炒糖色并翻炒香料后，加入开水炖煮。**老抽**是主要上色调料。
*   **简易做法**：冰糖融化后，加入**生抽10ml、老抽15ml**与五花肉一同翻炒至上色。
*   **湖南家常做法**：翻炒上色后，加入**生抽、老抽**等调料再翻炒。
*   **徽派做法**：加水炖煮一段时间后，加入**生抽、老抽、蚝油**继续煮。

### 总结与建议
*   **主要上色源**：**糖色（焦糖化）** 提供红亮色泽和焦糖风味，**老抽** 提供深红褐色。二者结合效果最佳。
*   **操作要点**：
    1.  **炒糖色时务必控制火候**（多用中小火），避免炒焦发苦。
    2.  **上色翻炒**：糖色炒好后，需将五花肉倒入锅中充分翻炒，使其均匀裹上糖色。
    3.  **酱油加入时机**：通常在上色翻炒时或加水炖煮前加入，以便颜色渗入。
*   **选择**：如果您是新手，可从**简易做法**或**湖南家常做法**开始，步骤相对直接。如果追求更专业的风味和色泽，可尝试**南派做法**的专用糖色工艺。

**请注意**：所有做法均强调“收汁”阶段需注意火候，避免收干，以保留浓稠汤汁包裹在肉块上，使成品色泽油亮诱人。

😊😊😊
回答是合理的。 LLM还矫正了料酒不上色，并没有幻觉

In [139]:
display_md_jupyter(rag('如何制作无骨鸡爪'))

distance:0.0164, content:主题: 无骨鸡爪的做法 > 章节: 操作 > 细节: 去骨
内容: 这一步可以省略，此步骤大约花费 2 小时  
- 放入冰箱，**冷冻层** 20 分钟
- 把全部放入不是冷冻层的冰箱，然后分批**10个一批**拿出来去骨
- 从手指（鸡爪的）最前端开始，每只手指都要用刀划开**划到它的手背部分**
- 再从手背部用刀分划开至整个手臂
- 把每只手指关节处都掰一掰**按手指出声音时那种**
- 按着它的手指最前端，往里推，每只手指都一样，先推到中间手掌手背部分
distance:0.0161, content:主题: 无骨鸡爪的做法 > 章节: 必备原料和工具 > 细节: 无
内容: - 鸡爪
- 姜
- 料酒
- 大葱
- 大蒜
- 小米辣
- 洋葱
- 生抽
- 蚝油
- 黑醋（推荐陈醋）
- 白糖
- 盐
- 花椒油
- 香菜
- 柠檬
- 折耳根
distance:0.0159, content:# 无骨鸡爪的做法

![无骨鸡爪成品](./无骨鸡爪.jpg)
**图片里的颜色比较浅，家里人爱吃酱油少的**

这是一道做法简单但消耗体力和耐力的无骨鸡爪，酸辣开胃，Q 弹爽口，第一次做的话总耗时 8 个小时 15 分钟。

预估烹饪难度：★★★★★

## 必备原料和工具

- 鸡爪
- 姜
- 料酒
- 大葱
- 大蒜
- 小米辣
- 洋葱
- 生抽
- 蚝油
- 黑醋（推荐陈醋）
- 白糖
- 盐
- 花椒油
- 香菜
- 柠檬
- 折耳根

## 计算

一份正好够 2-3 个人吃。

- 鸡爪 1kg
- 姜 4 片
- 料酒 65g
- 大葱 3 段（5cm 一段）
- 大蒜 10 瓣
- 小米辣 4 个少辣，6 个中辣，12 个大辣（推荐大辣）
- 洋葱 （半个）
- 生抽 75g = 15g * 5
- 蚝油 30g = 15g * 2
- 黑醋（推荐陈醋） 50g
- 白糖 10g
- 盐 3g
- 花椒油 10ml
- 香菜 3 颗
- 柠檬 2 颗（以 1 颗为单位来调整酸度）
- 折耳根 10g

## 操作

### 鸡爪处理

- 用剪刀 / 刀 把鸡爪上的指甲的部分全部剪掉 **包括指甲下面的肉和骨头，让它一点指甲都不剩**
- 用水把他们洗干净，放一边
- 把`鸡爪`放入大锅中，

根据提供的上下文，以下是制作无骨鸡爪的完整方法：

### **无骨鸡爪的做法**

**预估烹饪难度：** ★★★★★
**总耗时：** 约 8 小时 15 分钟（含腌制时间）
**份量：** 2-3 人

---

#### **一、必备原料和工具**
*   **原料：** 鸡爪 1kg、姜 4 片、料酒 65g、大葱 3 段、大蒜 10 瓣、小米辣 4-12 个、洋葱半个、生抽 75g、蚝油 30g、黑醋（推荐陈醋） 50g、白糖 10g、盐 3g、花椒油 10ml、香菜 3 颗、柠檬 2 颗、折耳根 10g。
*   **工具：** 剪刀或刀、煮锅、冰箱、容器。

#### **二、操作步骤**
**1. 鸡爪处理（去腥与预煮）**
*   用剪刀或刀将鸡爪的指甲部分完全剪掉（**包括指甲下面的肉和骨头**），确保一点指甲都不剩。
*   清洗干净。
*   将鸡爪放入大锅，加入大葱段、料酒和姜片，加水没过鸡爪。
*   大火煮开，中途撇去浮沫。**保持沸腾（100℃）10分钟**。
*   关火后捞出鸡爪，沥干水分，再次洗净，放入盆中。

**2. 去骨（此步骤可省略，约耗时2小时）**
*   将煮好的鸡爪放入冰箱**冷冻层**20分钟。
*   移至冰箱冷藏层，然后**分批次（每次约10个）** 取出去骨。
*   **去骨手法：**
    *   从鸡爪每根手指的最前端开始，用刀划开直至手背部分。
    *   再从手背处用刀划开至整个手臂（鸡爪的腕部）。
    *   将每根手指的关节处掰动，直至发出“咔”的响声。
    *   按住每根手指的最前端，向里推至手掌手背处。
    *   当每根手指的皮肉分离后，从手掌开始向手臂方向推，直至整张皮肉完全脱离骨头。
*   将去骨的鸡爪放入碗中备用。

**3. 调配腌料与腌制**
*   将小米辣切均匀小颗；大蒜、洋葱、香菜、折耳根切碎。
*   将柠檬对半切开，把柠檬汁挤入装有鸡爪的容器。
*   将**所有调料**（生抽、蚝油、黑醋、白糖、盐、花椒油等）以及切好的小米辣、大蒜、洋葱、香菜、折耳根全部倒入鸡爪容器中。
*   抓拌均匀。
*   放入冰箱冷藏**一个晚上（约6小时）** 入味。

#### **三、附加内容**
*   煮鸡爪时需注意观察水位，如低于食材的3/4，应加热水补充。
*   可以参考视频教程学习去骨技巧：[bili_89324373958](https://www.bilibili.com/video/BV1t44y117D8?share_source=copy_web)。

**请注意：** 此菜谱耗时较长，尤其是去骨步骤需要耐心。如果觉得去骨太繁琐，可以省略该步骤，制作成有骨的酸辣鸡爪。

😊😊😊对于需要详细制作的。LLM只是让语句变流畅了，没有改变

In [143]:
display_md_jupyter(rag('给我推荐最高难度川菜'))

distance:0.0299, content:主题: 2 星难度菜品 > 章节: 无 > 细节: 无
内容: * [鲣鱼海苔玉米饭](./../dishes/staple/鲣鱼海苔玉米饭/鲣鱼海苔玉米饭.md)
* [麻辣减脂荞麦面](./../dishes/staple/麻辣减脂荞麦面.md)
* [凉拌木耳](./../dishes/vegetable_dish/凉拌木耳/凉拌木耳.md)
* [凉拌莴笋](./../dishes/vegetable_dish/凉拌莴笋/凉拌莴笋.md)
distance:0.0258, content:主题: 4 星难度菜品 > 章节: 无 > 细节: 无
内容: * [空气炸锅照烧鸡饭](./../dishes/staple/空气炸锅照烧鸡饭/空气炸锅照烧鸡饭.md)
* [蒸卤面](./../dishes/staple/蒸卤面.md)
* [鲜肉烧卖](./../dishes/staple/鲜肉烧卖.md)
* [鹰嘴豆炸饼](./../dishes/staple/鹰嘴豆炸饼.md)
distance:0.0251, content:主题: 2 星难度菜品 > 章节: 无 > 细节: 无
内容: * [白灼菜心](./../dishes/vegetable_dish/白灼菜心/白灼菜心.md)
* [糖拌西红柿](./../dishes/vegetable_dish/糖拌西红柿/糖拌西红柿.md)
* [素炒豆角](./../dishes/vegetable_dish/素炒豆角.md)
distance:0.0164, content:# 8 星难度菜品

* [照烧鸡腿饭](./../dishes/staple/照烧鸡腿饭.md)
distance:0.0164, content:主题: 4 星难度菜品 > 章节: 无 > 细节: 无
内容: * [冷吃兔](./../dishes/meat_dish/冷吃兔.md)
* [咕噜肉](./../dishes/meat_dish/咕噜肉.md)
* [咖喱肥牛](./../dishes/meat_dish/咖喱肥牛/咖喱肥牛.md)
* [啤酒鸭](./../dishes/meat_dish/啤酒鸭/啤酒鸭.md)
distance:0.0161, content:##

根据提供的上下文，最高难度的菜品被标记为 **8 星难度**，但其中列出的唯一菜品是 **照烧鸡腿饭**，这并非传统意义上的川菜。

在较低难度（4 星）的菜品列表中，可以找到一些经典的 **川菜**，例如：
*   回锅肉
*   宫保鸡丁
*   小炒黄牛肉
*   尖叫牛蛙

**结论：** 上下文信息中，没有同时满足“最高难度”和“川菜”这两个条件的菜品。最高难度（8星）的菜品中没有川菜，而川菜中在上下文里难度最高的是 4 星菜品。

😶😊是合理的，因为本身没有答案，这个只要菜在知识库。由LLM选择

In [144]:
display_md_jupyter(rag('番茄有什么营养价值'))

I0000 00:00:1777734281.334428    8657 chttp2_transport.cc:1182] unix:/tmp/tmpai5t93rp_milvus_cookrag.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {created_time:"2026-05-02T23:04:41.334413018+08:00", http2_error:11, grpc_status:14}
E0000 00:00:1777734281.334548    8657 chttp2_transport.cc:1210] unix:/tmp/tmpai5t93rp_milvus_cookrag.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


distance:0.0164, content:主题: 番茄红酱的做法 > 章节: 必备原料和工具 > 细节: 无
内容: - 碎牛肉
- 蒜瓣
- 胡萝卜
- 芹菜
- 洋葱
- 橄榄油
- 糖
- 食盐
- 胡椒粉
- 番茄酱
- 牛奶
- 干罗勒或百里香（可选）
distance:0.0164, content:主题: 利提巧卡的做法 > 章节: 无 > 细节: 无
内容: > Litti Chokha(लिट्टी चोखा)— 比哈尔邦烤麦球配蔬菜泥  
利提巧卡是印度比哈尔邦(Bihar)最具代表性的传统主食。"利提"是用全麦面粉包裹烤鹰嘴豆粉(Sattu)馅料烤制而成的面球，"巧卡"是将茄子、番茄、土豆等蔬菜烧烤后捣碎的蘸酱。这是一道古老的食物，曾是士兵们行军时的干粮。制作过程需要耐心，约 60 分钟完成。  
预估烹饪难度：★★★★★
distance:0.0161, content:# 电饭煲蒸米饭的做法

预估烹饪难度：★

## 必备原料和工具

* 电饭煲
* 江南米或北方大米
* 水

## 计算

* 一般一个人可以食用 100ml-200ml 的米。

指尖量水法 （用于精准确定米水量的方法）

* 江南米，米和水放在电饭煲的容器内，食指触及米时，水量能刚好没过食指的第一个指节又第二个指节 1/4 处（即大约 2.6 厘米）。
* 北方大米，米和水放在电饭煲的容器内，食指触及米，水量能刚好没过食指的第一个指节（即大约 2 厘米处）。

使用上述条件，计算出计划使用的原材料比例。

## 操作

* 清洗米
* 将米和水一起加入电饭煲中。
* 连接电饭煲电源，进入加热模式。等待大约 30 分钟。
* 待电饭煲自动进入保温模式后。
* 将米在电饭煲中闷 10-15 分钟。
* 盛出米。

## 附加内容

- 技术总结
  - 不论什么容器，水的量都是可以使用“指尖量水法”煮出好米饭。
  - 水的多少是能够严重影响一碗好米饭的口感。
  - 南方一般食用江南米，北方一般食用北方大米，江南米会比较细长，而北方大米会比较短且圆。
  - 米饭不同品种的米，不同的水量会极度影响口感。
- 更多信息请参考使用的电饭煲的说明书。

如果您遵循本指南的制作流程而发现有问题或可以改进的流程，请提出 Issue 或 Pull request 。

di

根据提供的上下文信息，我无法回答番茄的具体营养价值。上下文主要包含各种菜谱的制作方法、原料用量和操作步骤，虽然在“番茄红酱的做法”中提到其“营养丰富”，但并未提供任何具体的营养成分、含量或健康益处的详细信息。

作为专业的烹饪饮食专家，我的回答必须严格基于给定的上下文。因此，对于“番茄有什么营养价值”这一问题，我的回答是：**不知道**。

如果您需要了解番茄的营养学详情，建议查阅专业的营养学资料或食品成分数据库。

😊😶😊可以看到，LLM理解了上下文，出于prompt，他有着明显的限制